# Granular energy balance parsing for melt contributing time periods

In [ ]:
# Need to separate
# term
# elevation: maximum 3, otherwise too many bars on one plot (3 *12 = 36 bars)
# time period: monthly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.cbook import boxplot_stats
import pandas as pd
import xarray as xr
import seaborn as sns

In [ ]:
%reload_ext autoreload
%autoreload 2

# Read in pre-computed energy balance zarr datasets

In [ ]:
# Read in the em zarr file
basin = 'animas'
wy = 2022
zarr_dir = '/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/zarr_stores'
em_zarr_fn = f'{zarr_dir}/{basin}_unified_em_wy{wy}.zarr'
em_ds = xr.open_zarr(em_zarr_fn, chunks="auto")
em_ds

In [ ]:
# Pull in net solar radiation to separate out SW and LW components of net radiation
# Load the net_solar zarr (built separately by build_zarr_datacube.py)
net_solar_ds = xr.open_zarr(f'{zarr_dir}/{basin}_unified_net_solar_wy{wy}.zarr')

# Assign net_solar and derive net_LW from net_rad - net_solar, then drop net_rad
em_ds = (em_ds.assign(net_solar = net_solar_ds['net_solar'],
                      net_LW    = em_ds['net_rad'] - net_solar_ds['net_solar'],
                      )
                    #   .drop_vars('net_rad')
        )
em_ds

# Temporal aggregation

In [ ]:
cols = ['evaporation', 'latent_heat', 'net_solar', 'net_LW', 'precip_advected', 'sensible_heat', 'snow_soil', 'sum_EB']
rad_cols = ['net_solar', 'net_LW']
non_rad_cols = [var for var in cols if var not in rad_cols]

In [ ]:
# For each month plot the basin-averaged energy balance components
# Compress each term into monthly average value
em_ds_monthly = em_ds.resample(time='ME').mean().compute()

In [ ]:
# Pull out the month labels for the x-axis
time_labels = [str(t)[:7] for t in em_ds_monthly['time'].values]  # e.g. "2018-10"

# Compute boxplot stats for plotting
stats = {}
for i, t in enumerate(time_labels):
    stats[t] = {
        var: boxplot_stats(em_ds_monthly[var].isel(time=i).values.ravel())[0]
        for var in cols
    }

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Plot radiation separately form the rest of the terms
rad_cols = ['net_solar', 'net_LW']
non_rad_cols = [var for var in cols if var not in rad_cols]
n_vars = len(cols)

width = 0.8 / n_vars
offsets = np.linspace(-0.4, 0.4, n_vars)
for j, var in enumerate(non_rad_cols):
    positions = np.arange(len(time_labels)) + offsets[j]
    ax.bxp([stats[t][var] for t in time_labels],
           positions=positions, widths=width, showfliers=False,
        #    whiskerprops={'visible': False}, capprops={'visible': False},
           patch_artist=True, boxprops={'facecolor': plt.cm.Set2(j / n_vars)}, medianprops={'color': 'black'}
           )
ax.set_xticks(np.arange(len(time_labels)))
ax.set_xticklabels(time_labels, rotation=45)


legend_handles = [
    Patch(facecolor=plt.cm.Set2(j / n_vars), label=var)
    for j, var in enumerate(non_rad_cols)
]
ax.legend(handles=legend_handles)
plt.title(f'Monthly Average EB Components: {basin.capitalize()} WY {wy}')

# Now plot radiation and amend spacing
n_vars = len(rad_cols)
offsets = np.linspace(-0.1, 0.1, n_vars)

fig, ax = plt.subplots(figsize=(14, 5))
for j, var in enumerate(rad_cols):
    positions = np.arange(len(time_labels)) + offsets[j]
    ax.bxp([stats[t][var] for t in time_labels],
           positions=positions, widths=width, showfliers=False,
        #    whiskerprops={'visible': False}, capprops={'visible': False},
              patch_artist=True, boxprops={'facecolor': plt.cm.viridis(j / n_vars)}, medianprops={'color': 'black'}
           )

ax.set_xticks(np.arange(len(time_labels)))
ax.set_xticklabels(time_labels, rotation=45)
legend_handles = [
    Patch(facecolor=plt.cm.viridis(j / n_vars), label=var)
    for j, var in enumerate(rad_cols)
]
ax.legend(handles=legend_handles)
plt.title(f'Monthly Average Radiation: {basin.capitalize()} WY {wy}')

# Compress to basin-averaged values

In [ ]:
# Collapse the spatial dimensions to get basin-averaged values
em_ds_monthly_basin = em_ds_monthly.mean(dim=['x', 'y'])
df = em_ds_monthly_basin.to_pandas()
# redo time index to be month names
df.index = df.index.strftime('%b')
df

In [ ]:
cols = ['evaporation', 'latent_heat', 'net_solar', 'net_LW', 'precip_advected', 'sensible_heat', 'snow_soil']
df[cols].plot(kind='bar', stacked=True, colormap='Set2', figsize=(12, 6))
# rotate xtick labels
plt.xticks(rotation=0)
# Add horizontal zero line
plt.axhline(0, color='black', linewidth=0.8)
# Plot the sum_EB as a narrow bar on top of the stacked bars
plt.bar(df.index, df['sum_EB'], color='k', width=0.1, label='sum_EB')
# Make legend 2 columns
plt.legend(ncol=2)
plt.title(f'Basin-averaged Energy Balance Components by Month: {basin.capitalize()} WY {wy}')

# Plot radially, by month

In [ ]:
import sys
sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')

In [ ]:
import plot_sdd_radial as psr
import helpers as h


In [ ]:
def load_terrain(basin, script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts', verbose=False):
    terrain_fn = h.fn_list(script_dir, f'{basin}*_setup/data/{basin}_terrain.nc')[0]
    if verbose:
        print(f'{terrain_fn}\n')

    # Load the terrain and SDD shift datasets
    if verbose:
        print('Loading datasets...')
    terrain_ds = xr.open_dataset(terrain_fn, drop_variables=['spatial_ref', 'hs', 'slope'])
    return terrain_ds

In [ ]:
def plot_eb_term_radial(em_ds_monthly, terrain_ds, var, time_idx,
                        elev_min=None, elev_max=None, bin_size=200,
                        num_aspect_bins=8, verbose=False,
                        figsize=None, fontsize=None,
                        elev_labels_on=True, show_cbar=True,
                        cmap=None, vmin=None, vmax=None,
                        title=None, outname=None, ax=None):
    """Radial plot of a monthly-averaged EB term binned by aspect and elevation.

    Parameters
    ----------
    em_ds_monthly : xr.Dataset   monthly-averaged EM dataset
    terrain_ds    : xr.Dataset   must contain 'dem' and 'aspect'
    var           : str          EB variable name (e.g. 'net_solar')
    time_idx      : int          index into em_ds_monthly time dimension
    elev_min/max  : float        elevation range (defaults to DEM extent)
    bin_size      : float        elevation bin size in metres (default 200)
    vmin/vmax     : float        colorbar limits (defaults to symmetric ±abs_max)
    """
    # Pull the single monthly slice and merge with terrain
    em_slice = em_ds_monthly[[var]].isel(time=time_idx)
    combo_ds = xr.merge([terrain_ds[['dem', 'aspect']], em_slice])

    # Build elevation bins from user-specified range or DEM extent
    dem_vals = combo_ds['dem'].values
    lo = elev_min if elev_min is not None else np.floor(np.nanmin(dem_vals) / bin_size) * bin_size
    hi = elev_max if elev_max is not None else np.ceil(np.nanmax(dem_vals) / bin_size) * bin_size
    elevation_bins = np.arange(lo, hi + bin_size, bin_size)

    # Symmetric colorbar limits if not provided
    if vmin is None or vmax is None:
        flat = em_slice[var].values.ravel()
        lim = np.nanpercentile(np.abs(flat), 95)
        vmin, vmax = -lim, lim

    t_label = str(em_ds_monthly['time'].isel(time=time_idx).values)[:7]
    title = title or f'{t_label}'
    aspect_labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'][:num_aspect_bins]

    psr.plot_radial(
        combo_ds=combo_ds,
        combo_var=var,
        combo_var_units='W m⁻²',
        elevation_bins=elevation_bins,
        num_aspect_bins=num_aspect_bins,
        aspect_labels=aspect_labels,
        figsize=figsize, fontsize=fontsize,
        elev_labels_on=elev_labels_on,
        cmap=cmap,
        vminnorm=vmin, vmaxnorm=vmax,
        title=title,
        ax=ax, show_cbar=show_cbar,
        outname=outname,
        verbose=verbose
    )
    plt.suptitle(f'{var}')


In [ ]:
# Set elevation bins, consistent across WSC 3 basins and melt plot bin steps
bin_step = 200
elev_bins = np.arange(1200, 4400 + bin_step, bin_step)

# Choose a single month to test - use the first timestep
month_idx = 0

# Choose a single term
term = 'net_solar'

# Plot the monthly average up
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(em_ds_monthly[term].isel(time=month_idx), cmap='viridis')
plt.colorbar(im, ax=ax, label=f'{term} (units)')
ax.set_title(f'{term} for {time_labels[month_idx]}')

# now compute and plot the radial map
terrain_ds = load_terrain(basin=basin)
plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=month_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=9, elev_labels_on=False
                        )

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
# Set elevation bins, consistent across WSC 3 basins and melt plot bin steps
bin_step = 200
elev_bins = np.arange(1200, 4400 + bin_step, bin_step)

# now compute and plot the radial map
terrain_ds = load_terrain(basin=basin)

In [ ]:
# Plot them all as a grid of radial maps
nrows = 3
ncols = 4

# Choose a single term
term = 'net_solar'

# compute cbar lims
all_vals = np.concatenate([
    em_ds_monthly[term].isel(time=i).values.ravel()
    for i in range(em_ds_monthly.sizes['time'])
])
lim = np.nanpercentile(np.abs(all_vals), 95)
vmin, vmax = -lim, lim
print(lim)

fig, axes = plt.subplots(nrows, ncols,
                         subplot_kw={'projection': 'polar'},
                         figsize=(ncols * 5, nrows * 5),
                         constrained_layout=True
                         )

for ax, time_idx in zip(axes.ravel(), range(len(time_labels))):
    plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=time_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=11, elev_labels_on=False,
                        ax=ax, show_cbar=False,
                        )

fig.colorbar(cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax), cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True)),
             ax=axes.ravel().tolist(), aspect=30,
             orientation='vertical', pad=0.02, label=f'{var} (W m⁻²)')

In [ ]:
# Plot them all as a grid of radial maps
# Choose a single term
term = 'net_LW'

# compute cbar lims
all_vals = np.concatenate([
    em_ds_monthly[term].isel(time=i).values.ravel()
    for i in range(em_ds_monthly.sizes['time'])
])
lim = np.nanpercentile(np.abs(all_vals), 95)
vmin, vmax = -lim, lim
print(lim)

fig, axes = plt.subplots(nrows, ncols,
                         subplot_kw={'projection': 'polar'},
                         figsize=(ncols * 5, nrows * 5),
                         constrained_layout=True
                         )

for ax, time_idx in zip(axes.ravel(), range(len(time_labels))):
    plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=time_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=11, elev_labels_on=False,
                        ax=ax, show_cbar=False,
                        )

fig.colorbar(cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax), cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True)),
             ax=axes.ravel().tolist(), aspect=30,
             orientation='vertical', pad=0.02, label=f'{var} (W m⁻²)')

In [ ]:
em_ds_monthly.data_vars

In [ ]:
# Plot them all as a grid of radial maps
# Choose a single term
term = 'sensible_heat'

# compute cbar lims
all_vals = np.concatenate([
    em_ds_monthly[term].isel(time=i).values.ravel()
    for i in range(em_ds_monthly.sizes['time'])
])
lim = np.nanpercentile(np.abs(all_vals), 95)
vmin, vmax = -lim, lim
print(lim)

fig, axes = plt.subplots(nrows, ncols,
                         subplot_kw={'projection': 'polar'},
                         figsize=(ncols * 5, nrows * 5),
                         constrained_layout=True
                         )

for ax, time_idx in zip(axes.ravel(), range(len(time_labels))):
    plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=time_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=11, elev_labels_on=False,
                        ax=ax, show_cbar=False,
                        )

fig.colorbar(cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax), cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True)),
             ax=axes.ravel().tolist(), aspect=30,
             orientation='vertical', pad=0.02, label=f'{var} (W m⁻²)')

In [ ]:
# Plot them all as a grid of radial maps
# Choose a single term
term = 'latent_heat'

# compute cbar lims
all_vals = np.concatenate([
    em_ds_monthly[term].isel(time=i).values.ravel()
    for i in range(em_ds_monthly.sizes['time'])
])
lim = np.nanpercentile(np.abs(all_vals), 95)
vmin, vmax = -lim, lim
print(lim)

fig, axes = plt.subplots(nrows, ncols,
                         subplot_kw={'projection': 'polar'},
                         figsize=(ncols * 5, nrows * 5),
                         constrained_layout=True
                         )

for ax, time_idx in zip(axes.ravel(), range(len(time_labels))):
    plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=time_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=11, elev_labels_on=False,
                        ax=ax, show_cbar=False,
                        )

fig.colorbar(cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax), cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True)),
             ax=axes.ravel().tolist(), aspect=30,
             orientation='vertical', pad=0.02, label=f'{var} (W m⁻²)')

In [ ]:
# Plot them all as a grid of radial maps
# Choose a single term
term = 'snow_soil'

# compute cbar lims
all_vals = np.concatenate([
    em_ds_monthly[term].isel(time=i).values.ravel()
    for i in range(em_ds_monthly.sizes['time'])
])
lim = np.nanpercentile(np.abs(all_vals), 95)
vmin, vmax = -lim, lim
print(lim)

fig, axes = plt.subplots(nrows, ncols,
                         subplot_kw={'projection': 'polar'},
                         figsize=(ncols * 5, nrows * 5),
                         constrained_layout=True
                         )

for ax, time_idx in zip(axes.ravel(), range(len(time_labels))):
    plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=time_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=11, elev_labels_on=False,
                        ax=ax, show_cbar=False,
                        )

fig.colorbar(cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax), cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True)),
             ax=axes.ravel().tolist(), aspect=30,
             orientation='vertical', pad=0.02, label=f'{var} (W m⁻²)')

In [ ]:
# Plot them all as a grid of radial maps
# Choose a single term
term = 'sum_EB'

# compute cbar lims
all_vals = np.concatenate([
    em_ds_monthly[term].isel(time=i).values.ravel()
    for i in range(em_ds_monthly.sizes['time'])
])
lim = np.nanpercentile(np.abs(all_vals), 95)
vmin, vmax = -lim, lim
print(lim)

fig, axes = plt.subplots(nrows, ncols,
                         subplot_kw={'projection': 'polar'},
                         figsize=(ncols * 5, nrows * 5),
                         constrained_layout=True
                         )

for ax, time_idx in zip(axes.ravel(), range(len(time_labels))):
    plot_eb_term_radial(em_ds_monthly, terrain_ds, var=term, time_idx=time_idx,
                        elev_min=elev_bins[0], elev_max=elev_bins[-1], bin_size=bin_step,
                        cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True),
                        figsize=(4,3), fontsize=11, elev_labels_on=False,
                        ax=ax, show_cbar=False,
                        )

fig.colorbar(cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax), cmap=sns.palettes.color_palette('RdYlBu_r', as_cmap=True)),
             ax=axes.ravel().tolist(), aspect=30,
             orientation='vertical', pad=0.02, label=f'{var} (W m⁻²)')

In [ ]:
em_ds_monthly.data_vars

In [ ]:
em_ds_monthly['sensible_heat'].isel(time=-3).plot()